# Laboratorium 5: sieci projektowe — CPM, PERT i wykresy Gantta

[![Otwórz w Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AdoHaha/automation_robotics_lab/blob/master/pl/lab_auto_5.ipynb)

W laboratorium 4 planowaliśmy operacje w komórce robotycznej. Było to **harmonogramowanie typu job-shop**: zadania konkurowały o fizyczne stanowiska, więc zmienne binarne wybierały kolejność ich obsługi.

Teraz patrzymy na cały projekt. Zespół robotyki ma przygotować działający pokaz. Pytamy:

- Które zadania projektowe mogą przebiegać równolegle?
- Które wyznaczają najwcześniejszy termin ukończenia?
- Które opóźnienia nie wpływają na zakończenie, a które przesuwają cały projekt?
- Kiedy warto zapłacić za szybszą dostawę lub dodatkowe ograniczenie ryzyka?

Wykorzystamy:

- **metodę ścieżki krytycznej (CPM)** dla ustalonych czasów zadań,
- **PERT** dla niepewnych czasów zadań,
- **wykresy Gantta** do przedstawiania harmonogramu i zapasu czasu.

## Cele ćwiczenia

Po wykonaniu zadań potrafisz:

1. zapisać opis projektu robotycznego jako zadania, czasy i ograniczenia kolejności,
2. obliczyć najwcześniejsze i najpóźniejsze początki/końce, zapas czasu i ścieżkę krytyczną,
3. wyjaśnić, dlaczego przyspieszenie zadania niekrytycznego może oszczędzić **zero** dni projektu,
4. narysować i odczytać projektowy wykres Gantta,
5. wykorzystać oszacowania PERT do przybliżonej oceny ryzyka niedotrzymania terminu,
6. porównać szybsze dostawy, skracanie czasu zadań i działania zmniejszające ryzyko.

## Scenariusz: pokaz autonomicznego robota inspekcyjnego

Laboratorium przygotowuje autonomicznego robota inspekcyjnego na dzień otwarty. Robot ma poruszać się w niewielkiej komórce produkcyjnej, sprawdzić część i przekazać wynik gościom.

Kierownictwo chce, jeśli to możliwe, skrócić plan deterministyczny do **30 dni**. Zespół może:

1. wybrać szybszego, droższego dostawcę wybranych podzespołów,
2. skrócić wybrane prace inżynierskie dzięki dodatkowym osobom lub nadgodzinom,
3. dodać krótki przegląd zmniejszający ryzyko, który na wykresie Gantta wygląda jak dodatkowa praca, lecz może ograniczyć późniejszą niepewność.

Pytanie brzmi: „Która decyzja skraca **ścieżkę krytyczną** lub zmniejsza ryzyko na tyle, by uzasadnić koszt?”. Samo przyspieszenie pojedynczego zadania nie wystarcza.

## Widok projektu

![Planowanie projektu autonomicznego robota inspekcyjnego](https://raw.githubusercontent.com/AdoHaha/automation_robotics_lab/refs/heads/master/lab_auto_5_illustration.png)

Ta sama ilustracja znajduje się w głównym katalogu repozytorium jako `lab_auto_5_illustration.png`, jeśli środowisko korzysta z lokalnych plików.

## Rozgrzewka na papierze: sieć czterech zadań

Przed uruchomieniem Pythona rozwiąż przykład ręcznie. Poznaj cztery oznaczenia czasów CPM, zanim przeczytasz implementację.

- **ES** (*earliest start*, najwcześniejszy początek): najwcześniejszy dzień rozpoczęcia po zakończeniu wszystkich poprzedników.
- **EF** (*earliest finish*, najwcześniejszy koniec): `ES + duration`.
- **LF** (*latest finish*, najpóźniejszy koniec): najpóźniejszy dzień zakończenia bez opóźniania projektu.
- **LS** (*latest start*, najpóźniejszy początek): `LF - duration`.
- **Zapas czasu** (`slack`): `LS - ES`. Zerowy zapas oznacza zadanie krytyczne.

| Zadanie | Czas trwania | Poprzednicy |
|:---:|---:|:---|
| A | 2 | — |
| B | 3 | — |
| C | 5 | A, B |
| D | 4 | C |

Odpowiedz bez kodu:

1. Jakie są ES i EF każdego zadania?
2. Jakie są LS i LF, jeśli projekt kończy się możliwie najwcześniej?
3. Które zadania mają zerowy zapas czasu?
4. Jaka jest ścieżka krytyczna?
5. Czy skrócenie A o 1 dzień przyspiesza projekt? Dlaczego?

Wskazówki:

- W przejściu w przód zadania bez poprzedników rozpoczynają się w dniu 0.
- Gdy zadanie ma kilku poprzedników, jego ES wyznacza największe EF poprzednika.
- Najwcześniejszy koniec projektu to EF ostatniego zadania.
- W przejściu wstecz zaczynamy od najwcześniejszego końca projektu i cofamy się przez sieć.
- Zadanie jest krytyczne, gdy zapas czasu wynosi zero. Ścieżka krytyczna musi tworzyć połączony łańcuch od początku do końca projektu.
- Aby ocenić skrócenie A, sprawdź, czy to A wyznacza początek C.

## Importy i konfiguracja środowiska

Notebook działa w Google Colab i lokalnym Jupyterze. Uruchom następną komórkę. Jeśli brakuje pakietu, komórka instaluje potrzebne biblioteki w bieżącym środowisku notebooka.

In [ ]:
# W Colab i lokalnym Jupyterze używamy tych samych bibliotek; instalujemy je po nieudanym imporcie.
try:
    from itertools import product
    from math import erf, sqrt

    import matplotlib.pyplot as plt
    import networkx as nx
    import numpy as np
    import pandas as pd
except ModuleNotFoundError:
    # W Colab pakiety instalują się w środowisku notebooka.
    # Lokalnie najlepiej użyć środowiska wirtualnego i `pip install -r requirements.txt`.
    %pip install numpy pandas matplotlib networkx

    from itertools import product
    from math import erf, sqrt

    import matplotlib.pyplot as plt
    import networkx as nx
    import numpy as np
    import pandas as pd


## Dane głównego projektu

Czasy są podane w **dniach**. Kolumna `predecessors` zawiera zadania, które muszą się zakończyć przed rozpoczęciem danego zadania.

Kolumny `a`, `m` i `b` to oszacowania PERT:

- `a`: czas optymistyczny,
- `m`: czas najbardziej prawdopodobny,
- `b`: czas pesymistyczny.

W CPM najpierw wykorzystamy ustaloną kolumnę `duration`. W PERT obliczymy czas oczekiwany na podstawie `a`, `m` i `b`.

Nazwy kolumn i kluczy pozostają zgodne z kodem: `id` — identyfikator, `task` — opis zadania, `owner` — odpowiedzialny zespół, `duration` — czas trwania. `ES`, `EF`, `LS`, `LF` oznaczają czasy CPM, `slack` — zapas czasu, `critical` — oznaczenie zadania krytycznego, a `successors` — następniki.

In [ ]:
# Przechowuj czasy deterministyczne i PERT razem, aby obie analizy korzystały z tych samych zależności.
project_tasks = pd.DataFrame([
    {
        "id": "A",
        "task": 'Określenie wymagań pokazu',
        "owner": 'Kierownik projektu',
        "duration": 2,
        "predecessors": [],
        "a": 1,
        "m": 2,
        "b": 3,
    },
    {
        "id": "B",
        "task": 'Zatwierdzenie bezpieczeństwa obszaru testów',
        "owner": 'Specjalista ds. bezpieczeństwa',
        "duration": 4,
        "predecessors": ["A"],
        "a": 3,
        "m": 4,
        "b": 7,
    },
    {
        "id": "C",
        "task": 'Architektura oprogramowania i interfejsy',
        "owner": 'Zespół oprogramowania',
        "duration": 4,
        "predecessors": ["A"],
        "a": 3,
        "m": 4,
        "b": 7,
    },
    {
        "id": "D",
        "task": 'Dostawa czujnika LiDAR',
        "owner": 'Dział zakupów',
        "duration": 8,
        "predecessors": ["A"],
        "a": 5,
        "m": 8,
        "b": 13,
    },
    {
        "id": "E",
        "task": 'Dostawa modułu akumulatora',
        "owner": 'Dział zakupów',
        "duration": 5,
        "predecessors": ["A"],
        "a": 3,
        "m": 5,
        "b": 9,
    },
    {
        "id": "F",
        "task": 'Budowa podwozia',
        "owner": 'Zespół mechaniczny',
        "duration": 6,
        "predecessors": ["A"],
        "a": 4,
        "m": 6,
        "b": 9,
    },
    {
        "id": "G",
        "task": 'Przygotowanie scenariusza symulacji',
        "owner": 'Zespół symulacji',
        "duration": 4,
        "predecessors": ["B", "C"],
        "a": 3,
        "m": 4,
        "b": 6,
    },
    {
        "id": "H",
        "task": 'Montaż elektroniki i układu percepcji',
        "owner": 'Zespół integracji',
        "duration": 5,
        "predecessors": ["D", "E", "F"],
        "a": 4,
        "m": 5,
        "b": 9,
    },
    {
        "id": "I",
        "task": 'Integracja oprogramowania nawigacji',
        "owner": 'Zespół oprogramowania',
        "duration": 6,
        "predecessors": ["C", "H"],
        "a": 4,
        "m": 6,
        "b": 10,
    },
    {
        "id": "J",
        "task": 'Stanowiskowe testy bezpieczeństwa',
        "owner": 'Specjalista ds. bezpieczeństwa',
        "duration": 4,
        "predecessors": ["B", "I"],
        "a": 3,
        "m": 4,
        "b": 8,
    },
    {
        "id": "K",
        "task": 'Test w komórce robotycznej',
        "owner": 'Zespół robotyki',
        "duration": 5,
        "predecessors": ["G", "J"],
        "a": 3,
        "m": 5,
        "b": 11,
    },
    {
        "id": "L",
        "task": 'Scenariusz pokazu i szkolenie operatora',
        "owner": 'Asystent prowadzącego',
        "duration": 3,
        "predecessors": ["A"],
        "a": 2,
        "m": 3,
        "b": 4,
    },
    {
        "id": "M",
        "task": 'Końcowa próba pokazu',
        "owner": 'Cały zespół',
        "duration": 2,
        "predecessors": ["K", "L"],
        "a": 1,
        "m": 2,
        "b": 4,
    },
    {
        "id": "N",
        "task": 'Gotowość na dzień otwarty',
        "owner": 'Kierownik projektu',
        "duration": 1,
        "predecessors": ["M"],
        "a": 1,
        "m": 1,
        "b": 2,
    },
])

project_tasks


## Implementacja CPM

CPM wykonuje dwa przejścia przez skierowany graf acykliczny. Kierunek krawędzi to: poprzednik → następnik.

### Przejście w przód

Najwcześniejszy początek zadania `i` wyznacza poprzednik kończący się najpóźniej. Bez poprzedników przyjmujemy `ES_i = 0`.

$$
ES_i = \max_{p \in \operatorname{pred}(i)} EF_p
$$

$$
EF_i = ES_i + \operatorname{duration}_i
$$

### Przejście wstecz

Zaczynamy od najwcześniejszego końca projektu. Cofając się przez sieć, wyznaczamy najpóźniejszy koniec zadania `i` na podstawie następnika, który musi rozpocząć się najwcześniej. Bez następników przyjmujemy `LF_i` równy końcowi projektu.

$$
LF_i = \min_{s \in \operatorname{succ}(i)} LS_s
$$

$$
LS_i = LF_i - \operatorname{duration}_i
$$

Zapas czasu określa, o ile zadanie można opóźnić bez przesunięcia końca projektu:

$$
\operatorname{slack}_i = LS_i - ES_i
$$

Zadania z zerowym zapasem czasu są krytyczne. Opóźnienie takiego zadania zwykle przesuwa cały projekt, chyba że późniejsza decyzja zmieni sieć.

### Działanie funkcji pomocniczych: wiersze zadań → wyniki CPM

Funkcje przyjmują DataFrame zadań: `validate_project_data` sprawdza identyfikatory i odwołania, `build_project_graph` buduje acykliczny graf kolejności, a `compute_cpm` wykonuje oba przejścia czasowe. Zwrócone tabela, graf i dzień zakończenia trafiają do `critical_edges` i `one_critical_path`, które identyfikują łańcuchy krytyczne na podstawie zerowego zapasu **oraz ciągłości czasowej**. `duration_col` wybiera czasy ustalone lub oczekiwane; tolerancje pozwalają rozpoznać niewielkie różnice numeryczne.

In [ ]:
# Oddziel sprawdzanie grafu, obliczenia czasów i wyznaczanie ścieżki, aby móc obejrzeć każdy wynik.
def validate_project_data(tasks):
    """Sprawdza identyfikatory zadań i odwołania do poprzedników przed budową grafu.

    tasks to DataFrame pandas z kolumną unikalnych identyfikatorów `id`
    i kolumną `predecessors` zawierającą listy identyfikatorów. Zwraca None
    przy poprawnych danych; zgłasza ValueError przy duplikatach lub odwołaniu
    do nieistniejącego zadania. Nie sprawdza czasów ani cykli; cykle sprawdza
    build_project_graph. Iteruje przez pandas bez zmiany danych wejściowych."""
    ids = list(tasks["id"])
    if len(ids) != len(set(ids)):
        raise ValueError('Identyfikatory zadań muszą być unikalne.')

    known_ids = set(ids)
    missing = []
    for row in tasks.itertuples(index=False):
        for predecessor in row.predecessors:
            if predecessor not in known_ids:
                missing.append((row.id, predecessor))
    if missing:
        raise ValueError(f"Odwołania do nieznanych poprzedników: {missing}")


def build_project_graph(tasks):
    """Buduje skierowaną sieć kolejności dla CPM i symulacji.

    tasks jest DataFrame z unikalnymi `id` i listami `predecessors`.
    Sprawdza odwołania, tworzy NetworkX DiGraph i dodaje krawędzie od
    poprzednika do następnika. Zwraca graf; zgłasza ValueError dla duplikatów,
    nieznanych odwołań lub cyklu. Wywołuje validate_project_data i korzysta
    z zaimportowanego aliasu nx biblioteki networkx. Nie zmienia tasks."""
    validate_project_data(tasks)
    graph = nx.DiGraph()
    graph.add_nodes_from(tasks["id"])

    for row in tasks.itertuples(index=False):
        for predecessor in row.predecessors:
            graph.add_edge(predecessor, row.id)

    # CPM wymaga acyklicznej sieci kolejności.
    if not nx.is_directed_acyclic_graph(graph):
        cycle = nx.find_cycle(graph)
        raise ValueError(f"Sieć projektu zawiera cykl: {cycle}")
    return graph


def compute_cpm(tasks, duration_col="duration", tolerance=1e-9):
    """Oblicza czasy zadań i koniec projektu przejściami w przód i wstecz.

    tasks to niepusty DataFrame z `id`, listami `predecessors` i liczbowymi
    czasami w kolumnie wskazanej przez duration_col (domyślnie "duration").
    Oczekuje nieujemnych czasów w dniach; tolerance domyślnie wynosi 1e-9
    dnia i służy do klasyfikacji zerowego zapasu czasu.
    Zwraca (table, graph, project_duration): kopia tabeli zachowuje kolumny
    wejściowe i dodaje duration_used, ES, EF, LS, LF, slack, critical oraz
    successors; graph to sprawdzony NetworkX DiGraph, a czas jest w dniach.
    Wywołuje build_project_graph, korzysta z pandas/NetworkX. Nie wyświetla
    wyników i nie zmienia tabeli wejściowej."""
    tasks = tasks.copy(deep=True)
    graph = build_project_graph(tasks)
    order = list(nx.topological_sort(graph))
    duration = dict(zip(tasks["id"], tasks[duration_col]))

    # Przejście w przód: zadanie zaczyna się po zakończeniu wszystkich poprzedników.
    earliest_start = {}
    earliest_finish = {}
    for task_id in order:
        predecessor_finishes = [earliest_finish[p] for p in graph.predecessors(task_id)]
        earliest_start[task_id] = max(predecessor_finishes) if predecessor_finishes else 0
        earliest_finish[task_id] = earliest_start[task_id] + duration[task_id]

    project_duration = max(earliest_finish.values())

    # Przejście wstecz: cofaj się od końca projektu do pierwszych zadań.
    # Zadanie z następnikami musi zakończyć się przed najmniejszym z ich najpóźniejszych początków.
    latest_finish = {}
    latest_start = {}
    for task_id in reversed(order):
        successor_starts = [latest_start[s] for s in graph.successors(task_id)]
        latest_finish[task_id] = min(successor_starts) if successor_starts else project_duration
        latest_start[task_id] = latest_finish[task_id] - duration[task_id]

    # Zachowaj opisy zadań obok czasów do tabel, diagramów i późniejszych porównań scenariuszy.
    result = tasks.set_index("id").loc[order].reset_index()
    result["duration_used"] = result[duration_col]
    result["ES"] = result["id"].map(earliest_start)
    result["EF"] = result["id"].map(earliest_finish)
    result["LS"] = result["id"].map(latest_start)
    result["LF"] = result["id"].map(latest_finish)
    result["slack"] = result["LS"] - result["ES"]
    result["critical"] = result["slack"].abs() <= tolerance
    result["successors"] = result["id"].apply(lambda task_id: list(graph.successors(task_id)))
    return result, graph, project_duration


def critical_edges(cpm_table, graph, tolerance=1e-7):
    """Znajduje krawędzie łączące zadania krytyczne bez przerwy czasowej.

    cpm_table to wynik compute_cpm z `id`, `critical`, `ES`, `EF`;
    graph to odpowiadający mu NetworkX DiGraph z tymi samymi ID.
    tolerance domyślnie wynosi 1e-7 dnia przy porównaniu EF poprzednika
    z ES następnika. Zwraca listę krotek (source_id, target_id) należących
    do łańcuchów krytycznych. Korzysta z czasów CPM: nie każda krawędź
    między czerwonymi węzłami jest krytyczna. Nie wyświetla i nie zmienia danych."""
    metrics = cpm_table.set_index("id")
    critical_nodes = set(cpm_table.loc[cpm_table["critical"], "id"])
    edges = []
    for source, target in graph.edges():
        if source in critical_nodes and target in critical_nodes:
            # Węzły krytyczne tworzą ciągłą ścieżkę tylko wtedy, gdy koniec poprzednika
            # jest równy początkowi następnika; inaczej występuje przerwa.
            touches_without_gap = abs(metrics.loc[source, "EF"] - metrics.loc[target, "ES"]) <= tolerance
            if touches_without_gap:
                edges.append((source, target))
    return edges


def one_critical_path(cpm_table, graph, tolerance=1e-7):
    """Zwraca jeden połączony łańcuch krytyczny od początku do końca projektu.

    cpm_table i graph pochodzą z compute_cpm; tabela wymaga `id`, `ES`,
    `EF` i `critical`. tolerance domyślnie wynosi 1e-7 dnia przy sprawdzaniu
    końców i przerw. Buduje podgraf przez critical_edges, wylicza ścieżki
    od początku do końca i zwraca listę ID o największej sumie czasów.
    Zwraca pustą listę, jeśli brak takiego łańcucha. Korzysta z NetworkX
    i critical_edges; podaje jedną ścieżkę, nie wszystkie równorzędne."""
    # Sama krytyczność zadań nie wystarcza: szukaj połączonych ścieżek między początkiem a końcem projektu.
    metrics = cpm_table.set_index("id")
    project_duration = cpm_table["EF"].max()

    critical_graph = nx.DiGraph()
    critical_nodes = list(cpm_table.loc[cpm_table["critical"], "id"])
    critical_graph.add_nodes_from(critical_nodes)
    critical_graph.add_edges_from(critical_edges(cpm_table, graph, tolerance=tolerance))

    starts = [
        node for node in critical_graph.nodes
        if critical_graph.in_degree(node) == 0 and abs(metrics.loc[node, "ES"]) <= tolerance
    ]
    ends = [
        node for node in critical_graph.nodes
        if critical_graph.out_degree(node) == 0 and abs(metrics.loc[node, "EF"] - project_duration) <= tolerance
    ]

    best_path = []
    best_duration = -1
    durations = (metrics["EF"] - metrics["ES"]).to_dict()
    for start in starts:
        for end in ends:
            for path in nx.all_simple_paths(critical_graph, start, end):
                path_duration = sum(durations[node] for node in path)
                if path_duration > best_duration:
                    best_duration = path_duration
                    best_path = path
    return best_path


## Bazowy wynik CPM

Uruchom CPM dla deterministycznej kolumny `duration`.

In [ ]:
# Użyj zwróconego grafu i tabeli czasów także na późniejszych wykresach.
baseline_cpm, baseline_graph, baseline_duration = compute_cpm(project_tasks, duration_col="duration")

# Tabela zawiera dane potrzebne do interpretacji CPM.
# Wyznacz ścieżkę samodzielnie na podstawie zerowego zapasu czasu i kolejności.
baseline_cpm[[
    "id", "task", "duration", "predecessors", "successors", "ES", "EF", "LS", "LF", "slack"
]]


### Zadanie 1 — interpretacja wariantu bazowego

Odpowiedz w notebooku przed uruchomieniem wizualizacji:

1. Jaki jest najwcześniejszy możliwy dzień zakończenia projektu? Użyj największego `EF` lub `baseline_duration`.
2. Które zadania mają zerowy zapas czasu?
3. Połącz je w poprawną ścieżkę krytyczną za pomocą kolumn poprzedników/następników. Sama lista niepołączonych zadań nie jest ścieżką.
4. Które zadanie niekrytyczne ma największy zapas i o ile dni może się opóźnić bez zmiany końca projektu?
5. Która dostawa jest groźniejsza przy opóźnieniu: LiDAR-u czy akumulatora? Uzasadnij zapasem czasu, a nie intuicją o podzespołach.
6. Dlaczego ścieżka krytyczna projektu to inna idea niż zajęte stanowisko z laboratorium 4?

## Wizualizacja sieci i wykres Gantta

Diagram sieci pokazuje kolejność. Wykres Gantta pokazuje najwcześniejsze początki, długości zadań i zapas czasu.

Ten wykres różni się od wykresu harmonogramowania typu job-shop z laboratorium 4: wiersz to **zadanie projektowe**, a nie fizyczne stanowisko. Zadania mogą nakładać się w czasie, jeśli pozwalają na to ograniczenia kolejności.

### Funkcje wykresów: wyniki CPM → widoki harmonogramu

`plot_project_network(cpm_table, graph, title)` rozmieszcza zadania według najwcześniejszego początku i wyróżnia połączone krawędzie krytyczne. `plot_project_gantt(cpm_table, deadline, title)` sortuje zadania, rysuje pracę i okna zapasu czasu oraz opcjonalnie linię terminu. Obie funkcje zwracają `(fig, ax)`, aby umożliwić zmianę podpisów i wyglądu. Nie zmieniają czasów wejściowych i nie dodają ograniczeń zasobowych.

In [ ]:
# Oba widoki korzystają z wyników CPM; wizualizacja nie zmienia harmonogramu.
def plot_project_network(cpm_table, graph, title='Sieć projektu'):
    """Tworzy diagram kolejności rozmieszczony według najwcześniejszego początku.

    cpm_table to tabela compute_cpm z `id`, `ES`, `duration_used`,
    `slack`, `critical`; graph musi zawierać te same ID zadań. title to
    napis, domyślnie "Sieć projektu". Rozsuwa pionowo zadania o tym samym
    ES, podpisuje czas i zapas w dniach, wyróżnia critical_edges na czerwono.
    Zwraca obiekty Matplotlib (fig, ax) do wyświetlenia lub modyfikacji.
    Korzysta z pandas, NetworkX, Matplotlib i critical_edges; diagram
    nie nakłada fizycznych ograniczeń zasobowych."""
    table = cpm_table.set_index("id")

    # Rozmieść węzły według ES; zadania o podobnym początku rozsuń pionowo.
    groups = cpm_table.groupby("ES")["id"].apply(list).to_dict()
    pos = {}
    for x, task_ids in groups.items():
        offset = (len(task_ids) - 1) / 2
        for j, task_id in enumerate(task_ids):
            pos[task_id] = (float(x), -(j - offset))

    # Wyróżnij krytyczność kolorem, pozostawiając czas i zapas w etykietach.
    node_colors = ["#d62728" if table.loc[node, "critical"] else "#8ecae6" for node in graph.nodes]
    edge_colors = ["#d62728" if edge in critical_edges(cpm_table, graph) else "#999999" for edge in graph.edges]
    edge_widths = [2.8 if edge in critical_edges(cpm_table, graph) else 1.2 for edge in graph.edges]
    labels = {
        node: f"{node}\n{table.loc[node, 'duration_used']:g} dni\nzapas {table.loc[node, 'slack']:g}"
        for node in graph.nodes
    }

    fig, ax = plt.subplots(figsize=(13, 7))
    nx.draw_networkx_edges(
        graph, pos, ax=ax, edge_color=edge_colors, width=edge_widths,
        arrows=True, arrowsize=16, connectionstyle="arc3,rad=0.05"
    )
    nx.draw_networkx_nodes(
        graph, pos, ax=ax, node_color=node_colors,
        node_size=1800, edgecolors="black", linewidths=1
    )
    nx.draw_networkx_labels(graph, pos, labels=labels, ax=ax, font_size=8)
    ax.set_title(title)
    ax.set_xlabel('Najwcześniejszy dzień rozpoczęcia')
    ax.set_yticks([])
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.set_axisbelow(True)
    plt.tight_layout()
    return fig, ax


def plot_project_gantt(cpm_table, deadline=None, title='Wykres Gantta projektu'):
    """Tworzy wykres Gantta najwcześniejszych początków z oknami zapasu czasu.

    cpm_table to tabela compute_cpm z `id`, `task`, `ES`, `EF`,
    `duration_used`, `slack`, `critical`; wszystkie czasy są w dniach.
    deadline to opcjonalny numer dnia projektu (None pomija linię);
    title domyślnie wynosi "Wykres Gantta projektu". Sortuje zadania,
    rysuje pracę i zapas, zwraca Matplotlib (fig, ax) do wyświetlenia lub zmian.
    Korzysta z pandas, NumPy i Matplotlib. Nakładanie się zadań wynika
    tylko z kolejności; nie sprawdza jednoczesnych prac tego samego zespołu."""
    plot_table = cpm_table.sort_values(["ES", "EF", "id"], ascending=[True, True, True]).reset_index(drop=True)
    y_positions = np.arange(len(plot_table))

    fig, ax = plt.subplots(figsize=(13, 0.55 * len(plot_table) + 2.5))
    for y, row in zip(y_positions, plot_table.itertuples(index=False)):
        task_color = "#d62728" if row.critical else "#1f77b4"
        ax.barh(y, row.duration_used, left=row.ES, color=task_color, edgecolor="black")
        ax.text(row.ES + row.duration_used / 2, y, row.id, ha="center", va="center", color="white", weight="bold")

        if row.slack > 1e-9:
            ax.barh(y, row.slack, left=row.EF, color="#cccccc", edgecolor="#666666", alpha=0.55, hatch="//")
            ax.text(row.EF + row.slack / 2, y, f"zapas {row.slack:g}", ha="center", va="center", fontsize=8)

    # Termin służy porównaniu; nie przesuwa narysowanych zadań.
    if deadline is not None:
        ax.axvline(deadline, linestyle="--", linewidth=2, color="black", label=f"termin: dzień {deadline}")
        ax.legend(loc="lower right")

    ax.set_yticks(y_positions)
    ax.set_yticklabels(plot_table["task"])
    ax.invert_yaxis()
    ax.set_xlabel('Dzień projektu')
    ax.set_title(title)
    ax.grid(axis="x", linestyle=":", alpha=0.4)
    ax.set_axisbelow(True)
    plt.tight_layout()
    return fig, ax


In [ ]:
# Wyróżnienie łańcuchów pomaga połączyć zadania o zerowym zapasie w ścieżki.
plot_project_network(baseline_cpm, baseline_graph, title='Bazowa sieć projektu: ścieżka krytyczna na czerwono');


In [ ]:
# Porównaj najwcześniejsze czasy z celem kierownictwa; wykres nie uwzględnia ograniczeń zasobowych.
plot_project_gantt(baseline_cpm, deadline=30, title='Bazowy wykres Gantta projektu');


### Zadanie 2 — odczytaj wykres Gantta

Kolorowy pasek przedstawia pracę, a szary kreskowany — zapas czasu. Odpowiedz na podstawie wykresu:

1. Które zadania mogą rozpocząć się bezpośrednio po A?
2. Które kończą się wcześnie, lecz nie wyznaczają końca projektu?
3. Czy opóźnienie L o 10 dni przesunie koniec projektu? Porównaj z zapasem L.
4. Czy opóźnienie D o 2 dni przesunie koniec projektu? Porównaj z zapasem D.
5. Które dwa lub trzy zapasy czasu warto przekazać kierownikowi projektu i dlaczego?

## Warstwa decyzyjna 1: dwóch dystrybutorów

Dział zakupów otrzymał dwie niezależne oferty dostaw ekspresowych. Dotyczą różnych części:

- zadanie D — dostawa czujnika LiDAR,
- zadanie E — dostawa modułu akumulatora.

Standardowy czas dostawy znajduje się już w harmonogramie bazowym. Dostawa ekspresowa jest szybsza, a `extra_cost` oznacza dopłatę do opcji standardowej. Pomijamy bazowe ceny zakupów, ponieważ są takie same w każdym scenariuszu i nie wpływają na decyzję.

Kusząca reguła „kup każdą szybszą opcję” jest błędna. CPM podpowiada:

> Zapłać za przyspieszenie, gdy wpływa ono na ścieżkę, która jest lub może stać się krytyczna.

Pytamy więc nie tylko o dni dostawy, lecz także o zaoszczędzone dni projektu.

Stałe etykiety wariantów w tabelach pozostają angielskie: `standard suppliers` — dostawy standardowe, `express LiDAR` / `express LiDAR only` — ekspresowy LiDAR, `express battery` / `express battery only` — ekspresowy akumulator, `express both` — obie dostawy ekspresowe.

In [ ]:
# Dni zaoszczędzone na dostawie są lokalną korzyścią; następny model sprawdza wpływ na cały projekt.
supplier_options = pd.DataFrame([
    {
        "component": 'Czujniki LiDAR',
        "task_id": "D",
        "standard_duration": 8,
        "express_duration": 5,
        "extra_cost": 2400,
        "comment": 'Specjalistyczny dystrybutor czujników z priorytetową wysyłką',
    },
    {
        "component": 'Moduły akumulatorów',
        "task_id": "E",
        "standard_duration": 5,
        "express_duration": 3,
        "extra_cost": 1200,
        "comment": 'Lokalny dystrybutor akumulatorów z wysyłką w tym samym tygodniu',
    },
])

supplier_options["delivery_days_saved"] = (
    supplier_options["standard_duration"] - supplier_options["express_duration"]
)
supplier_options["cost_per_delivery_day_saved"] = (
    supplier_options["extra_cost"] / supplier_options["delivery_days_saved"]
)

supplier_options


### Funkcje dostaw: zmiany czasów → oszczędność czasu projektu

`apply_duration_changes(tasks, changes)` kopiuje dane i zastępuje tylko wskazane czasy deterministyczne. `evaluate_duration_scenario(tasks, label, changes, extra_cost)` ponownie oblicza CPM, porównuje zakończenie z `baseline_duration` i zwraca słownik podsumowania, tabelę CPM oraz graf. Koszty są dopłatami w PLN; wynik określa koszt zaoszczędzonego dnia **projektu**, a nie dnia dostawy.

In [ ]:
# Oddziel scenariusze dostaw, aby ich zmiany nie kumulowały się w danych bazowych.
def apply_duration_changes(tasks, changes):
    """Kopiuje tabelę zadań i zastępuje wskazane czasy deterministyczne.

    tasks to DataFrame z `id` i liczbowym `duration` w dniach. changes
    przypisuje istniejącym ID nowe czasy w dniach; pusty słownik oznacza
    brak zmian. Zwraca kopię DataFrame z zastąpionymi czasami, zachowując
    zależności i oszacowania PERT. Zgłasza ValueError dla nieznanych ID.
    Korzysta z indeksowania pandas. Wywołujący podaje sensowne nieujemne
    czasy; późniejsze compute_cpm ocenia wpływ na projekt."""
    updated = tasks.copy(deep=True)
    for task_id, new_duration in changes.items():
        if task_id not in set(updated["id"]):
            raise ValueError(f"Nieznany identyfikator zadania: {task_id}")
        updated.loc[updated["id"] == task_id, "duration"] = new_duration
    return updated


def evaluate_duration_scenario(tasks, label, changes=None, extra_cost=0):
    """Porównuje deterministyczny wariant dostaw z zapisanym wariantem bazowym.

    tasks to DataFrame projektu używany przez compute_cpm; label jest nazwą
    scenariusza. changes to opcjonalny słownik ID→czas w dniach (None oznacza
    brak zmian), a extra_cost domyślnie 0 PLN dopłaty do zakupów standardowych.
    Stosuje zmiany, oblicza CPM i zwraca (summary_dict, cpm_table, graph).
    Podsumowanie zawiera czas, zaoszczędzone dni, dopłatę, PLN na zaoszczędzony
    dzień projektu (NaN przy braku oszczędności) i napis ścieżki krytycznej.
    Korzysta z globalnego baseline_duration oraz apply_duration_changes,
    compute_cpm i one_critical_path; nie wycenia bazowych podzespołów."""
    changes = changes or {}
    scenario_tasks = apply_duration_changes(tasks, changes)
    scenario_cpm, scenario_graph, scenario_duration = compute_cpm(scenario_tasks, duration_col="duration")
    # Porównaj oszczędność czasu całego projektu z wariantem bazowym, nie tylko czas dostawy.
    days_saved = baseline_duration - scenario_duration
    cost_per_project_day = extra_cost / days_saved if days_saved > 0 else np.nan
    summary = {
        "scenario": label,
        "project_duration": scenario_duration,
        "days_saved_vs_baseline": days_saved,
        "extra_cost": extra_cost,
        "cost_per_project_day_saved": cost_per_project_day,
        "critical_path": " -> ".join(one_critical_path(scenario_cpm, scenario_graph)),
    }
    return summary, scenario_cpm, scenario_graph


# Porównaj wszystkie opcje dostaw; jest ich na tyle mało, że można je jawnie wyliczyć.
supplier_scenarios = [
    ("standard suppliers", {}, 0),
    ("express LiDAR only", {"D": 5}, 2400),
    ("express battery only", {"E": 3}, 1200),
    ("express both", {"D": 5, "E": 3}, 3600),
]

supplier_results = []
for label, changes, cost in supplier_scenarios:
    summary, _, _ = evaluate_duration_scenario(project_tasks, label, changes, cost)
    supplier_results.append(summary)

supplier_results_table = pd.DataFrame(supplier_results)
supplier_results_table


### Zadanie 3 — wybór dostawy

Korzystaj z obu tabel: `supplier_options` pokazuje skrócenie dostawy, a `supplier_results_table` — wpływ na cały projekt.

1. Która dostawa ekspresowa skraca projekt?
2. Która go nie skraca? Uzasadnij zapasem czasu lub ścieżką krytyczną.
3. Dlaczego dostawa LiDAR-u szybsza o 3 dni nie musi oszczędzać 3 dni projektu?
4. Które zadanie lub ścieżka zyskuje znaczenie po przyspieszeniu LiDAR-u?
5. Czy zapłacisz za obie dostawy ekspresowe? Porównaj oszczędność czasu projektu i koszt, a nie tylko czas dostawy.

## Warstwa decyzyjna 2: skracanie czasu zadań

Skracanie czasu zadań (*crashing*) oznacza dodatkowy wydatek, nadgodziny lub dodatkowych pracowników w zamian za krótszy czas wykonania.

Nie polega na przyspieszaniu wszystkich długich zadań. Długie zadanie niekrytyczne może mieć taki zapas czasu, że jego skrócenie nie zmieni końca projektu.

In [ ]:
# Limity i dopłaty w PLN/dzień określają dozwolone warianty skracania zadań.
crash_options = pd.DataFrame([
    {
        "task_id": "F",
        "task": 'Budowa podwozia',
        "max_reduction": 2,
        "cost_per_day": 900,
        "note": 'Dodaj drugiego technika i użyj gotowych wyciętych płyt montażowych',
    },
    {
        "task_id": "H",
        "task": 'Montaż elektroniki i układu percepcji',
        "max_reduction": 2,
        "cost_per_day": 1400,
        "note": 'Nadgodziny zespołu integracji',
    },
    {
        "task_id": "I",
        "task": 'Integracja oprogramowania nawigacji',
        "max_reduction": 3,
        "cost_per_day": 1600,
        "note": 'Zaangażuj doświadczonego programistę z innego projektu',
    },
    {
        "task_id": "J",
        "task": 'Stanowiskowe testy bezpieczeństwa',
        "max_reduction": 1,
        "cost_per_day": 1700,
        "note": 'Weekendowy przegląd bezpieczeństwa',
    },
    {
        "task_id": "K",
        "task": 'Test w komórce robotycznej',
        "max_reduction": 2,
        "cost_per_day": 2000,
        "note": 'Zarezerwuj wieczorne testy i dodatkowego operatora',
    },
])

crash_options


### Funkcje skracania: dopuszczalne redukcje → warianty planu

`apply_crash_plan(tasks, reductions)` odejmuje wskazane dni na kopii tabeli. `evaluate_full_plan` najpierw zastępuje czasy dostaw, następnie stosuje redukcje i zwraca zmienione zadania, tabelę CPM, graf oraz dzień zakończenia. `enumerate_supplier_and_crash_plans(tasks, crash_options, deadline)` łączy cztery wbudowane opcje dostaw z każdą dozwoloną całkowitą redukcją dni i zwraca nieposortowany DataFrame planów z kosztami oraz informacją o spełnieniu terminu. Plan wybierasz, filtrując i porządkując tę tabelę.

In [ ]:
# Oceń każdy wariant według jego nowej ścieżki; zadania bazowo krytyczne mogą przestać takie być.
def apply_crash_plan(tasks, reductions):
    """Kopiuje tabelę zadań i odejmuje wskazane redukcje czasu.

    tasks ma `id` i liczbowe `duration`. reductions przypisuje istniejącym
    ID nieujemne redukcje w dniach; enumeracja podaje całkowite liczby dni.
    Zwraca kopię tabeli z czasami ograniczonymi od dołu do zera, bez zmiany
    innych kolumn. Zgłasza ValueError dla ujemnych redukcji lub nieznanych ID.
    Korzysta z pandas; wywołujący pilnuje limitów i kosztów crash_options.
    Oszacowania PERT pozostają bez zmian — nie są automatycznie skracane."""
    updated = tasks.copy(deep=True)
    base_duration = dict(zip(updated["id"], updated["duration"]))
    for task_id, reduction in reductions.items():
        if reduction < 0:
            raise ValueError('Redukcja czasu nie może być ujemna.')
        if task_id not in base_duration:
            raise ValueError(f"Nieznany identyfikator zadania: {task_id}")
        updated.loc[updated["id"] == task_id, "duration"] = max(0, base_duration[task_id] - reduction)
    return updated


def evaluate_full_plan(tasks, supplier_changes=None, crash_reductions=None):
    """Stosuje zmiany dostaw, następnie skracanie zadań i ponownie oblicza CPM.

    tasks to DataFrame z `id`, `duration` w dniach i `predecessors`.
    supplier_changes przypisuje ID nowe czasy; crash_reductions przypisuje
    ID redukcje w dniach. Oba domyślnie None, czyli brak zmian.
    Zwraca (updated_tasks, cpm_table, graph, duration_days). Korzysta z
    apply_duration_changes, apply_crash_plan i compute_cpm; koszt oblicza
    wywołujący, a kolumny PERT pozostają bez zmian."""
    supplier_changes = supplier_changes or {}
    crash_reductions = crash_reductions or {}
    # Najpierw zmień czas dostawy, następnie odejmij redukcję od uzyskanego czasu.
    updated = apply_duration_changes(tasks, supplier_changes)
    updated = apply_crash_plan(updated, crash_reductions)
    cpm_table, graph, duration = compute_cpm(updated, duration_col="duration")
    return updated, cpm_table, graph, duration


def enumerate_supplier_and_crash_plans(tasks, crash_options, deadline=30):
    """Ocenia wbudowane warianty dostaw i wszystkie dozwolone całodniowe redukcje.

    tasks to DataFrame projektu z zadaniami dostaw D/E. crash_options to
    DataFrame z `task_id`, `max_reduction` w całych dniach i `cost_per_day`
    w PLN/dzień. deadline domyślnie oznacza dzień projektu 30.
    Wylicza cztery ustalone warianty dostaw oraz każdą całkowitą redukcję
    od zera do limitu; oblicza CPM i dodaje dopłaty za dostawy i skracanie.
    Zwraca nieposortowany DataFrame z supplier_choice, crash_reductions,
    project_duration, days_saved_vs_baseline, total_extra_cost, meets_deadline
    i critical_path. Korzysta z globalnego baseline_duration, product
    z itertools, evaluate_full_plan i one_critical_path. Nie wybiera planu
    automatycznie i nie odczytuje osobnej tabeli ofert dostawców."""
    supplier_cases = [
        ("standard suppliers", {}, 0),
        ("express LiDAR", {"D": 5}, 2400),
        ("express battery", {"E": 3}, 1200),
        ("express both", {"D": 5, "E": 3}, 3600),
    ]

    # Zakresy całkowite zawierają brak redukcji i każdą dozwoloną liczbę dni do limitu.
    crash_ranges = [range(int(row.max_reduction) + 1) for row in crash_options.itertuples(index=False)]
    crash_task_ids = list(crash_options["task_id"])
    cost_per_day = dict(zip(crash_options["task_id"], crash_options["cost_per_day"]))

    rows = []
    for supplier_label, supplier_changes, supplier_cost in supplier_cases:
        for values in product(*crash_ranges):
            reductions = dict(zip(crash_task_ids, values))
            _, cpm_table, graph, duration = evaluate_full_plan(
                tasks,
                supplier_changes=supplier_changes,
                crash_reductions=reductions,
            )
            # Uwzględnij tylko dopłaty za dostawy i skracanie, pomijając w rankingu bazowe ceny zakupów.
            crash_cost = sum(reductions[task_id] * cost_per_day[task_id] for task_id in crash_task_ids)
            total_cost = supplier_cost + crash_cost
            rows.append({
                "supplier_choice": supplier_label,
                "crash_reductions": reductions,
                "project_duration": duration,
                "days_saved_vs_baseline": baseline_duration - duration,
                "total_extra_cost": total_cost,
                "meets_deadline": duration <= deadline,
                "critical_path": " -> ".join(one_critical_path(cpm_table, graph)),
            })

    # Zwróć kolejność wyliczania; sposób filtrowania i sortowania wybierasz w zadaniu.
    return pd.DataFrame(rows)


In [ ]:
# Najpierw wygeneruj warianty; samodzielnie odfiltruj je według terminu i uporządkuj według kosztu.
deadline = 30
plans = enumerate_supplier_and_crash_plans(project_tasks, crash_options, deadline=deadline)

print(f"Wygenerowano {len(plans)} wariantów planu. Pierwsze wiersze nie są rankingiem.")
plans[[
    "supplier_choice", "crash_reductions", "project_duration", "total_extra_cost", "meets_deadline"
]].head(8)


### Zadanie 4 — wybierz plan

Pokaz ma być gotowy w dniu 30. Tabela `plans` zawiera wszystkie kombinacje dostaw i całkowitych redukcji czasu z `crash_options`.

1. Odfiltruj wiersze, gdzie `meets_deadline` jest `True`, i posortuj je według `total_extra_cost`.
2. Znajdź najtańszy plan spełniający termin.
3. Dlaczego jest tańszy niż skracanie tylko oczywistych zadań krytycznych z wariantu bazowego? Wskazówka: po zmianie ścieżka krytyczna może się przesunąć.
4. Wpisz wybrane zmiany dostaw i redukcje czasu w następnej komórce.
5. Po zastosowaniu planu narysuj ponownie wykres Gantta.
6. Wskaż nową ścieżkę krytyczną. Czy różni się od bazowej?

In [ ]:
# Wybrane słowniki tworzą nowy harmonogram; wariant bazowy pozostaje dostępny do porównania.
# DO ZROBIENIA: wpisz do słowników plan wybrany w zadaniu 4.
# Najpierw pozostaw puste słowniki wariantu bazowego, potem zmień je i uruchom ponownie.
selected_supplier_changes = {}
selected_crash_reductions = {}

selected_tasks, selected_cpm, selected_graph, selected_duration = evaluate_full_plan(
    project_tasks,
    supplier_changes=selected_supplier_changes,
    crash_reductions=selected_crash_reductions,
)

print(f"Czas wybranego planu: {selected_duration:.0f} dni")
print(f"Spełnia termin w dniu {deadline}: {selected_duration <= deadline}")

selected_cpm[["id", "task", "duration", "ES", "EF", "LS", "LF", "slack"]]


In [ ]:
# Narysuj tę samą tabelę CPM, z której odczytujesz czas i zapas czasu wybranego planu.
plot_project_gantt(selected_cpm, deadline=deadline, title='Wykres Gantta wybranego planu');


## PERT: niepewne czasy i ryzyko niedotrzymania terminu

CPM zakłada ustalone czasy. PERT zastępuje pojedynczy czas trzema oszacowaniami:

$$
E = \frac{a + 4m + b}{6}
$$

$$
\sigma^2 = \left(\frac{b-a}{6}\right)^2
$$

`E` to oczekiwany czas zadania, a `sigma^2` — jego wariancja. Typowe przybliżenie projektu sumuje wariancje wzdłuż oczekiwanej ścieżki krytycznej i opisuje czas zakończenia rozkładem normalnym.

To przydatne, lecz ma ograniczenie: rzeczywiste czasy mogą spowodować zmianę ścieżki krytycznej.

### Funkcje PERT: trzy oszacowania → przybliżone prawdopodobieństwo

`add_pert_columns(tasks)` zwraca kopię tabeli ze średnimi i wariancjami zadań. `pert_summary(tasks, deadline)` wykonuje CPM dla średnich, sumuje wariancje na jednej oczekiwanej ścieżce krytycznej i wykorzystuje `normal_cdf(z)` do przybliżenia prawdopodobieństwa zakończenia przed terminem. Zwraca tabelę CPM, graf i słownik podsumowania. Czasy są w dniach, wariancje w dniach kwadratowych, a prawdopodobieństwo nie ma jednostki.

In [ ]:
# Oddziel niepewność zadań, czasy ścieżki i prawdopodobieństwo terminu, aby pokazać przybliżenie.
def add_pert_columns(tasks):
    """Kopiuje zadania i oblicza oczekiwane czasy oraz wariancje PERT.

    tasks to DataFrame z liczbowymi `a`, `m`, `b` w dniach; wywołujący
    powinien zapewnić czas optymistyczny <= najbardziej prawdopodobny
    <= pesymistyczny. Zwraca kopię z expected_duration w dniach i variance
    w dniach kwadratowych, zachowując czasy deterministyczne i inne kolumny.
    Korzysta z arytmetyki pandas; nie losuje harmonogramów ani nie sprawdza
    poprawności trzech oszacowań."""
    result = tasks.copy(deep=True)
    result["expected_duration"] = (result["a"] + 4 * result["m"] + result["b"]) / 6
    result["variance"] = ((result["b"] - result["a"]) / 6) ** 2
    return result


def normal_cdf(z):
    """Oblicza dystrybuantę standardowego rozkładu normalnego dla jednego z.

    z to skalarna liczba standaryzowana, bez jednostki. Zwraca skalarne
    prawdopodobieństwo P(Z <= z), korzystając z erf i sqrt z math.
    Służy przybliżeniu normalnemu PERT; nie modeluje zadań projektu
    ani nie wybiera terminu."""
    return 0.5 * (1 + erf(z / sqrt(2)))


def pert_summary(tasks, deadline):
    """Przybliża prawdopodobieństwo spełnienia terminu na jednej oczekiwanej ścieżce.

    tasks to DataFrame projektu z `id`, `predecessors` oraz oszacowaniami
    PERT `a`, `m`, `b` w dniach. deadline to liczbowy dzień projektu.
    Dodaje expected_duration/variance, oblicza CPM dla średnich, wybiera
    jedną ścieżkę krytyczną i sumuje jej wariancje. Zwraca (cpm_table, graph,
    summary_dict) z polami deadline, expected_project_duration, critical_path,
    critical_path_variance (dni kwadratowe), critical_path_std_dev (dni),
    z_value i approx_probability_finish_by_deadline (0..1).
    Wywołuje add_pert_columns, compute_cpm, one_critical_path i normal_cdf;
    zakłada niezależne czasy na ścieżce i pomija przejście na inną ścieżkę.
    Zerowa wariancja daje deterministyczne porównanie z terminem."""
    # CPM dla czasów oczekiwanych wyznacza ścieżkę przyjętą w tym przybliżeniu prawdopodobieństwa.
    pert_tasks = add_pert_columns(tasks)
    cpm_table, graph, expected_duration = compute_cpm(pert_tasks, duration_col="expected_duration")
    path = one_critical_path(cpm_table, graph)

    metrics = cpm_table.set_index("id")
    # Przybliżenie PERT: sumuj wariancje tylko na oczekiwanej ścieżce krytycznej.
    critical_path_variance = metrics.loc[path, "variance"].sum()
    critical_path_std = sqrt(critical_path_variance)

    # Unikaj dzielenia przez zero, gdy wszystkie oszacowania na ścieżce są deterministyczne.
    if critical_path_std > 0:
        z = (deadline - expected_duration) / critical_path_std
        probability = normal_cdf(z)
    else:
        z = np.inf if deadline >= expected_duration else -np.inf
        probability = float(deadline >= expected_duration)

    summary = {
        "deadline": deadline,
        "expected_project_duration": expected_duration,
        "critical_path": " -> ".join(path),
        "critical_path_variance": critical_path_variance,
        "critical_path_std_dev": critical_path_std,
        "z_value": z,
        "approx_probability_finish_by_deadline": probability,
    }
    return cpm_table, graph, summary


In [ ]:
# Przyjmij termin oceny ryzyka niezależny od celu skrócenia planu deterministycznego.
risk_deadline = 36
pert_cpm, pert_graph, pert_stats = pert_summary(project_tasks, deadline=risk_deadline)

# Najpierw obejrzyj tabelę PERT poszczególnych zadań w następnej komórce.
# Po zapisaniu własnego oszacowania odkomentuj ostatni wiersz, aby sprawdzić podsumowanie.
pert_stats_for_check = pd.DataFrame([pert_stats])
# pert_stats_for_check


In [ ]:
# Obejrzyj średnie i wariancje zadań przed interpretacją prawdopodobieństwa dla projektu.
pert_cpm[[
    "id", "task", "a", "m", "b", "expected_duration", "variance", "ES", "EF", "slack"
]]


### Zadanie 5 — interpretacja PERT

Najpierw obejrzyj tabelę zadań PERT. Gdy chcesz sprawdzić prawdopodobieństwo spełnienia terminu, wyświetl `pert_stats_for_check` z wcześniejszej komórki.

1. Które zadania mają największą niepewność? Sprawdź wariancję.
2. Które niepewne zadania należą do ścieżki krytycznej PERT? Użyj zerowego zapasu czasu i kolejności.
3. Jakie jest przybliżone prawdopodobieństwo ukończenia przed terminem przyjętym do oceny ryzyka?
4. Dlaczego wynik jest przybliżeniem, a nie gwarancją?

## Warstwa decyzyjna 3: dodanie zadania zmniejszającego ryzyko

Zespół integracji proponuje **jednodniowy przegląd interfejsów czujników** przed montażem elektroniki.

Wygląda jak dodatkowa praca, ponieważ dodaje widoczne zadanie do sieci. Ma jednak zapobiegać błędom połączeń i interfejsów oraz zwiększać przewidywalność montażu.

Założenia ćwiczenia:

- Dodaj zadanie `R`: przegląd interfejsów czujników, 1 dzień, po D, E i F.
- Zmień H tak, aby zależało od R zamiast bezpośrednio od D, E i F.
- Skróć deterministyczny czas H z 5 do 4 dni.
- Zmień oszacowania PERT zadania H z `(a=4, m=5, b=9)` na `(a=3, m=4, b=5)`.

To inny mechanizm niż zwykłe skracanie czasu zadań. Przegląd może zmniejszyć zarówno czas oczekiwany, jak i niepewność późniejszej pracy, mimo dodania nowego zadania do harmonogramu.

Etykiety porównania pozostają kluczami danych: `original` — projekt pierwotny, `with sensor interface review` — projekt z przeglądem interfejsów.

### Funkcja przeglądu: zmienione zależności → dane porównania

`add_sensor_interface_review(tasks)` kopiuje projekt, dodaje jednodniowe R po D/E/F, uzależnia H od R i zastępuje jego czasy deterministyczne/PERT. Zwraca zmienioną tabelę. Następne wywołania przekazują ją do CPM i PERT, tworząc `risk_comparison`; sama funkcja nie rozstrzyga, czy warto zapłacić za przegląd.

In [ ]:
# Zapisz przegląd jako zmianę zależności i oszacowań H, a następnie porównaj obie analizy czasowe.
def add_sensor_interface_review(tasks):
    """Kopiuje projekt i dodaje opisany przegląd interfejsów czujników.

    tasks to DataFrame z `id`, `task`, `owner`, `duration`, `predecessors`,
    `a`, `m`, `b`. Musi zawierać H i odwołania do D/E/F; R nie powinno
    już istnieć. Zastępuje poprzedników H przez [R], ustawia jego czasy
    deterministyczne/PERT na 4/(3,4,5) dni i dodaje jednodniowe R po D/E/F.
    Zwraca zmieniony DataFrame przez pandas; wpływ na harmonogram i ryzyko
    obliczają osobno compute_cpm i pert_summary."""
    updated = tasks.copy(deep=True)

    # Uzależnij H od R, aby przegląd był wymagany przez sieć kolejności, a nie tylko dopisany do listy.
    h_index = updated.index[updated["id"] == "H"][0]
    updated.at[h_index, "predecessors"] = ["R"]
    updated.loc[h_index, ["duration", "a", "m", "b"]] = [4, 3, 4, 5]

    review_task = pd.DataFrame([{
        "id": "R",
        "task": 'Przegląd interfejsów czujników',
        "owner": 'Zespół integracji',
        "duration": 1,
        "predecessors": ["D", "E", "F"],
        "a": 1,
        "m": 1,
        "b": 1,
    }])

    return pd.concat([updated, review_task], ignore_index=True)


mitigated_tasks = add_sensor_interface_review(project_tasks)
mitigated_cpm, mitigated_graph, mitigated_duration = compute_cpm(mitigated_tasks, duration_col="duration")
mitigated_pert_cpm, mitigated_pert_graph, mitigated_pert_stats = pert_summary(mitigated_tasks, deadline=risk_deadline)


# Porównaj koniec deterministyczny i niepewność razem; dodatkowa praca może ograniczać ryzyko.
risk_comparison = pd.DataFrame([
    {"case": "original", "deterministic_duration": baseline_duration, **pert_stats},
    {"case": "with sensor interface review", "deterministic_duration": mitigated_duration, **mitigated_pert_stats},
])

risk_comparison


In [ ]:
# Pokaż miejsce przeglądu w harmonogramie, zanim ocenisz, czy zmniejszenie ryzyka uzasadnia koszt.
plot_project_gantt(mitigated_cpm, deadline=deadline, title='Deterministyczny harmonogram z przeglądem interfejsów czujników');


### Zadanie 6 — decyzja o ograniczeniu ryzyka

Skorzystaj z `risk_comparison` i wykresu zmienionego harmonogramu.

1. Czy przegląd zmienia deterministyczny czas projektu?
2. Czy zmienia oczekiwany czas projektu według PERT?
3. Czy zmienia przybliżone prawdopodobieństwo ukończenia przed terminem oceny ryzyka?
4. Czy dodasz przegląd za 900 PLN? Uzasadnij decyzję ryzykiem harmonogramu i kosztem.
5. Czym ta decyzja różni się od prostego skrócenia zadania?

## Wyzwanie: symulacja Monte Carlo

Przybliżenie normalne PERT zakłada, że bieżąca ścieżka krytyczna pozostaje ścieżką wyznaczającą zakończenie. Symulacja sprawdza to założenie przez wielokrotne losowanie czasów i ponowne obliczanie czasu projektu.

Używamy przeskalowanego rozkładu beta z parametrami kształtu dobranymi tak, aby oczekiwana wartość losowanego czasu odpowiadała średniej PERT:

$$
E = \frac{a + 4m + b}{6}
$$

Symulacja nadal jest przybliżeniem, ale pokazuje ograniczenie: w różnych wylosowanych scenariuszach ścieżki krytyczne mogą się różnić.

### Funkcje symulacji: wylosowane dni zadań → rozkład czasu zakończenia

`sample_pert_beta_durations(tasks, rng, weight)` zwraca tablicę czasów zgodną z kolejnością wierszy. `project_duration_from_duration_vector` oblicza dzień zakończenia przejściem w przód, opcjonalnie korzystając z gotowego grafu i kolejności. `monte_carlo_project_duration(tasks, trials, seed)` powtarza kroki z odtwarzalnymi losowaniami i zwraca Series dni zakończenia. Dopiero z tej serii obliczamy średnie, kwantyle, częstości spełnienia terminu i wykresy.

In [ ]:
# Zachowaj kolejność między próbami, lecz losuj nowe czasy — koniec mogą wyznaczać różne ścieżki.
def sample_pert_beta_durations(tasks, rng, weight=4):
    """Losuje wektor czasów zgodny z wierszami z przeskalowanych rozkładów beta-PERT.

    tasks ma liczbowe `a`, `m`, `b` w dniach, gdzie a <= m <= b.
    rng to numpy.random.Generator przekazany przez wywołującego; weight
    domyślnie 4 daje średnią (a+4m+b)/6 każdego zadania o zmiennym czasie.
    Zwraca tablicę NumPy float w kolejności wierszy wejściowych. Zadania
    z a bliskim b są deterministyczne; pozostałe czasy to niezależne
    losowania beta przeskalowane do [a,b]. Korzysta z NumPy i zmienia stan
    rng; nie oblicza grafu ani terminu, nie sprawdza oszacowań wejściowych."""
    a = tasks["a"].to_numpy(dtype=float)
    m = tasks["m"].to_numpy(dtype=float)
    b = tasks["b"].to_numpy(dtype=float)

    sampled = np.empty(len(tasks), dtype=float)
    # Stałe czasy pomijają obliczenia parametrów beta wymagające dzielenia przez szerokość przedziału.
    deterministic = np.isclose(a, b)
    sampled[deterministic] = a[deterministic]

    variable = ~deterministic
    width = b[variable] - a[variable]
    alpha = 1 + weight * (m[variable] - a[variable]) / width
    beta = 1 + weight * (b[variable] - m[variable]) / width
    sampled[variable] = a[variable] + rng.beta(alpha, beta) * width
    return sampled


def project_duration_from_duration_vector(tasks, duration_values, graph=None, order=None):
    """Oblicza wylosowany czas zakończenia wyłącznie przejściem w przód.

    tasks zawiera `id` i listy `predecessors`; duration_values to sekwencja
    liczb w dniach zgodna z kolejnością wierszy. Opcjonalne graph i order
    oznaczają pasujący DiGraph i topologiczną kolejność ID; przy None
    tworzy je build_project_graph/NetworkX. Zwraca największy najwcześniejszy
    koniec w dniach. Ponowne użycie graph/order pozwala nie budować tej samej
    topologii w każdej próbie. Nie zwraca zapasów, prawdopodobieństw ani tabeli CPM."""
    # Osobne wywołanie buduje graf; symulacja wielokrotna przekazuje gotowy graf i kolejność.
    if graph is None:
        graph = build_project_graph(tasks)
    if order is None:
        order = list(nx.topological_sort(graph))

    duration = dict(zip(tasks["id"], duration_values))
    earliest_finish = {}
    for task_id in order:
        predecessor_finishes = [earliest_finish[p] for p in graph.predecessors(task_id)]
        earliest_start = max(predecessor_finishes) if predecessor_finishes else 0
        earliest_finish[task_id] = earliest_start + duration[task_id]
    return max(earliest_finish.values())


def monte_carlo_project_duration(tasks, trials=3000, seed=42):
    """Losuje pełne harmonogramy i zwraca ich czasy zakończenia projektu.

    tasks to niepusty DataFrame z `id`, `predecessors` i poprawnymi `a`,
    `m`, `b` w dniach. trials jest dodatnią liczbą całkowitą, domyślnie 3000;
    seed domyślnie 42 zapewnia odtwarzalne losowania NumPy. Raz buduje graf
    i kolejność, w każdej próbie losuje czasy i wykonuje przejście w przód,
    więc ścieżka wyznaczająca koniec może się zmieniać. Zwraca pandas Series
    o nazwie project_duration z jednym czasem w dniach na próbę. Korzysta z
    build_project_graph, sample_pert_beta_durations i
    project_duration_from_duration_vector. Nie rysuje i nie wybiera terminu;
    wyniki zależą od oszacowań oraz założeń modelu."""
    rng = np.random.default_rng(seed)
    graph = build_project_graph(tasks)
    order = list(nx.topological_sort(graph))
    finishes = np.empty(trials, dtype=float)

    # W każdej próbie oblicz końce według poprzedników zamiast utrwalać oczekiwaną ścieżkę krytyczną.
    for trial in range(trials):
        sampled = sample_pert_beta_durations(tasks, rng)
        finishes[trial] = project_duration_from_duration_vector(tasks, sampled, graph=graph, order=order)

    return pd.Series(finishes, name="project_duration")


In [ ]:
# Odtwarzalne losowania pozwalają porównywać modele bez przypadkowych zmian ziarna generatora.
sim_original = monte_carlo_project_duration(project_tasks, trials=3000, seed=1)
sim_mitigated = monte_carlo_project_duration(mitigated_tasks, trials=3000, seed=1)

simulation_summary = pd.DataFrame([
    {
        "case": "original",
        "mean": sim_original.mean(),
        "p50": sim_original.quantile(0.50),
        "p90": sim_original.quantile(0.90),
        "prob_finish_by_risk_deadline": (sim_original <= risk_deadline).mean(),
    },
    {
        "case": "with sensor interface review",
        "mean": sim_mitigated.mean(),
        "p50": sim_mitigated.quantile(0.50),
        "p90": sim_mitigated.quantile(0.90),
        "prob_finish_by_risk_deadline": (sim_mitigated <= risk_deadline).mean(),
    },
])

simulation_summary


In [ ]:
# Nałóż rozkłady czasu zakończenia i wspólny termin, aby pokazać więcej niż samą średnią.
fig, ax = plt.subplots(figsize=(10, 5))
ax.hist(sim_original, bins=30, alpha=0.55, label="projekt pierwotny")
ax.hist(sim_mitigated, bins=30, alpha=0.55, label="z przeglądem interfejsów czujników")
ax.axvline(risk_deadline, linestyle="--", color="black", label=f"termin oceny ryzyka: dzień {risk_deadline}")
ax.set_xlabel('Symulowany czas projektu [dni]')
ax.set_ylabel('Liczba prób')
ax.set_title('Symulacja Monte Carlo czasu projektu')
ax.legend()
plt.tight_layout()


### Pytania do wyzwania

1. Porównaj przybliżenie normalne PERT z Monte Carlo. Czy wyniki są bliskie?
2. Który wynik pokażesz kierownictwu i dlaczego?
3. Rozszerz symulację na wybrany plan dostaw i skracania zadań.
4. Znajdź plan dający co najmniej 80% symulowanego prawdopodobieństwa ukończenia przed terminem oceny ryzyka.

## Szablon raportu końcowego

Przygotuj krótki raport:

### 1. Bazowy CPM

- Czas projektu:
- Ścieżka krytyczna:
- Trzy zadania z największym zapasem czasu:
- Jedno zdanie wyjaśniające znaczenie zapasu:

### 2. Wybór dostaw

- Zalecane zmiany dostaw:
- Zaoszczędzone dni:
- Koszt:
- Dlaczego druga opcja jest lub nie jest przydatna:

### 3. Skracanie czasu zadań

- Zalecane redukcje:
- Nowy czas projektu:
- Łączny koszt:
- Nowa ścieżka krytyczna:

### 4. Ryzyko PERT

- Oczekiwany czas projektu:
- Prawdopodobieństwo spełnienia terminu:
- Główne źródło ryzyka:
- Zalecenie dotyczące przeglądu interfejsów czujników:

### 5. Refleksja

W 3–5 zdaniach wyjaśnij różnice między projektowym wykresem Gantta z tego laboratorium a wykresem stanowisk z laboratorium 4.